# ISIC 2024 — Colab training (A100)

This notebook runs the **same** `src/train.py` you already tested locally, but on the real `train-image.hdf5`.

**Before you start**
1. Runtime → Change runtime type → GPU → **A100** (not T4/L4 if you can avoid it).
2. Push the training-loop commit to GitHub first. This notebook **clones GitHub**, so if `src/train.py` is only on your laptop, Colab will get the old smoke-test script.
3. In Colab: the key icon (Secrets) → add `KAGGLE_API_TOKEN` with the same `KGAT_...` value as `~/.kaggle/access_token` on your Mac. Enable notebook access.
4. Keep this tab in the foreground. Colab kills idle sessions. Checkpoints go to **Google Drive** so a disconnect does not wipe the run.

Local dummy AUROC was meaningless (fake colored squares). Numbers here, on real images, are the first ones that count.

## 1. Mount Google Drive

Checkpoints (`best.pt`, `last.pt`) are written here so they survive a runtime reset.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/skin-disease-predictor")
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("checkpoints →", CHECKPOINT_DIR)

## 2. Kaggle auth

Uses Colab Secrets (`KAGGLE_API_TOKEN`). Same token format you already put in `~/.kaggle/access_token` locally.

You must already have joined [ISIC 2024](https://www.kaggle.com/competitions/isic-2024-challenge) and accepted the rules.

In [ ]:
from pathlib import Path

from google.colab import userdata

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
token_path = kaggle_dir / "access_token"
token_path.write_text(userdata.get("KAGGLE_API_TOKEN").strip())
token_path.chmod(0o600)
print("wrote", token_path, "(token not printed)")

## 3. Clone the repo and install extras

Colab already has PyTorch with CUDA. Do **not** `pip install -r requirements.txt` here — that can reinstall CPU torch and break the GPU.

If the clone fails, the GitHub repo is probably private: either make it public or add a GitHub token to the clone URL.

In [ ]:
from pathlib import Path

import torch

REPO_URL = "https://github.com/Suncho1208/skin-disease-predictor.git"
REPO_DIR = Path("/content/skin-disease-predictor")

if not (REPO_DIR / ".git").exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
%pip install -q kaggle timm albumentations

print("repo:", REPO_DIR)
print(
    "cuda:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu",
)

## 4. Download the real image bundle

Only `train-image.hdf5` (~2 GB). `train.csv` / `val.csv` already live in the repo (all 393 malignant + 20k benign, patient-level split). Do not re-run `prepare.py` unless you want a different split.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
HDF5_PATH = DATA_DIR / "train-image.hdf5"

if not HDF5_PATH.exists():
    !kaggle competitions download -c isic-2024-challenge -f train-image.hdf5 -p {DATA_DIR}
    zip_path = DATA_DIR / "train-image.hdf5.zip"
    if zip_path.exists():
        !unzip -o {zip_path} -d {DATA_DIR}
        zip_path.unlink()
else:
    print("already have", HDF5_PATH)

!ls -lh {DATA_DIR}

## 5. DataLoader smoke test

One batch from the real hdf5. If this fails, do not start training — the path or Kaggle file is wrong.

In [ ]:
!python src/train.py --hdf5 {HDF5_PATH} --smoke --batch-size 32

## 6. Short training run (2 epochs)

Goal: confirm loss moves and val AUROC is not ~0.5-random. This is the Day 1 evening sanity check, not the overnight run.

If CUDA OOM, drop `--batch-size` to 32. If it looks easy, try 128 next time.

In [ ]:
!python src/train.py \
  --hdf5 {HDF5_PATH} \
  --epochs 2 \
  --batch-size 64 \
  --num-workers 2 \
  --checkpoint-dir {CHECKPOINT_DIR}

## 7. Overnight / longer run

After the 2-epoch run looks sane (loss down, AUROC clearly above 0.5), re-run the cell below. Resume is not wired yet — this starts from ImageNet weights again. That is OK for a first longer run.

Leave the tab open. `last.pt` is rewritten every epoch on Drive.

In [ ]:
# Uncomment when you are ready for the longer run:
# !python src/train.py \
#   --hdf5 {HDF5_PATH} \
#   --epochs 8 \
#   --batch-size 64 \
#   --num-workers 2 \
#   --checkpoint-dir {CHECKPOINT_DIR}

## What to look for

| Signal | Meaning |
|---|---|
| `device=cuda` and an A100 name | GPU is actually on |
| `pos_weight` around 50 | Class imbalance weighting is using the real 20k split |
| train loss going down over 2 epochs | Optimization is working |
| val AUROC well above 0.5 | Not a random ranker; dummy local run was ~0.58 on fake images |
| `best.pt` in Drive | You can reset the runtime without losing the weights |

If val AUROC stays near 0.5 after 2 epochs, stop and debug (wrong labels, hdf5 mismatch, no GPU) instead of launching 8 more epochs.